In [11]:
# !pip install requests beautifulsoup4 pandas openpyxl -q

import os
import re
import json
import time
import random
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP

import requests
import pandas as pd
from bs4 import BeautifulSoup


# =======================
# CONFIG
# =======================
BASE_URL = "https://www.pitstoparabia.com"

OUT_XLSX = "pitstoparabia_tyres.xlsx"
CHECKPOINT_PATH = "pitstop_checkpoint.json"
URLCACHE_PATH = "pitstop_all_product_urls.json"

SLEEP = 0.35
VERBOSE_ITEMS = False

SAVE_EVERY_ROWS = 200  # промежуточное сохранение каждые N новых строк

# ВАЖНО: чтобы НЕ начинал заново, а продолжил/перепрошёл участок:
# None = использовать чекпоинт как есть
# число = принудительно начать с этого индекса в url-списке
RESUME_FROM_INDEX = 5880   # <-- поставь None, когда всё доберёшь


# =======================
# HTTP SESSION
# =======================
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
})


# =======================
# HELPERS
# =======================
def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", (x or "")).strip()

def normalize_size_to_195_65R15(size_text: str, fallback_from_url: str | None = None) -> str | None:
    """
    Приводим к виду: 195/65R15 (без индексов).
    Берём из текста на странице или (как фолбэк) из URL, если там есть ...-205-55-r16-...
    """
    t = clean_text(size_text)

    m = re.search(r"(\d{3})\s*/\s*(\d{2,3})\s*R\s*(\d{2})", t, flags=re.I)
    if m:
        return f"{m.group(1)}/{m.group(2)}R{m.group(3)}"

    if fallback_from_url:
        mu = re.search(r"-(\d{3})-(\d{2,3})-r(\d{2})\b", fallback_from_url, flags=re.I)
        if mu:
            return f"{mu.group(1)}/{mu.group(2)}R{mu.group(3)}"

    return None

def request_with_retries(url: str, timeout=45, max_retries=6):
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(url, timeout=timeout)
            return r
        except requests.exceptions.RequestException as e:
            sl = 1.2 + random.random() * 1.2
            print(f"NET ERR retry {attempt}/{max_retries} -> sleep {sl:.2f}s | {type(e).__name__}: {e}")
            time.sleep(sl)
    # последний раз — пусть падает исключением выше
    return SESSION.get(url, timeout=timeout)

def get_soup(url: str, sleep: float = 0.3) -> tuple[BeautifulSoup, int]:
    r = request_with_retries(url)
    print(f"    GET {url} -> {r.status_code}")

    # важно: НЕ raise_for_status здесь, чтобы мы могли обработать 403/404 “мягко”
    time.sleep(sleep)
    return BeautifulSoup(r.text, "html.parser"), r.status_code


# =======================
# PARSE PRODUCT PAGE (Variant A: актуальные товары с ценой)
# =======================
def parse_product_page(url: str) -> tuple[dict | None, int]:
    soup, status = get_soup(url, sleep=SLEEP)

    # 404 / 410 и т.п.
    if status == 404:
        return None, status

    # 403 — признак блокировки. Мы хотим остановиться и продолжить позже.
    if status == 403:
        return None, status

    # модель (название товара)
    model = None
    h1 = soup.select_one("h1")
    if h1:
        model = clean_text(h1.get_text(" ", strip=True))

    # бренд: часто в breadcrumb / или в логотипе. На product-page чаще есть alt у brand logo.
    brand = None
    brand_img = soup.select_one(".brand_container img[alt]") or soup.select_one("img[alt][src*='brands']")
    if brand_img and brand_img.get("alt"):
        brand = clean_text(brand_img["alt"])

    # год
    year = None
    year_node = soup.select_one('div[title*="Year of manufacture"], li div[title*="Year of manufacture"]')
    if year_node:
        m = re.search(r"\b(20\d{2})\b", year_node.get_text(" ", strip=True))
        if m:
            year = int(m.group(1))

    # размер (без индексов)
    size = None
    size_li = soup.select_one("li.size_block b")  # <li class="size_block"><b>205/55 R16</b>
    if size_li:
        size = normalize_size_to_195_65R15(size_li.get_text(" ", strip=True), fallback_from_url=url)
    if not size:
        # fallback: из URL
        size = normalize_size_to_195_65R15("", fallback_from_url=url)

    # цена (final)
    price = None
    price_node = soup.select_one('[data-price-type="finalPrice"][data-price-amount]') \
             or soup.select_one("#product-price-[data-price-amount]")  # на всякий случай
    if price_node and price_node.get("data-price-amount"):
        raw = price_node["data-price-amount"].strip()
        try:
            d = Decimal(raw).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
            price = str(d)
        except InvalidOperation:
            price = None
    if not price:
        # fallback по тексту
        ptxt = soup.select_one(".special-price .price, .price-box .special-price .price, .price-box .price")
        if ptxt:
            t = clean_text(ptxt.get_text(" ", strip=True)).replace("AED", "").strip().replace(" ", "")
            m = re.search(r"(\d+(?:\.\d+)?)", t)
            if m:
                try:
                    d = Decimal(m.group(1)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
                    price = str(d)
                except InvalidOperation:
                    price = None

    # если нет цены — это не “актуальный товар”
    if not price:
        return None, status

    row = {
        "size": size,
        "brand": brand or "UNKNOWN",
        "model": model or "",
        "year": year,
        "price": price,
        "url": url,
    }
    return row, status


# =======================
# PIVOT + SAVE
# =======================
def build_pivot_min_price(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()

    need = {"size", "brand", "price"}
    if not need.issubset(df_raw.columns):
        return pd.DataFrame()

    df = df_raw.copy()
    df["Price"] = pd.to_numeric(df["price"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df["Size"] = df["size"].astype(str).str.strip()
    df["Brand"] = df["brand"].astype(str).str.strip()

    df = df.dropna(subset=["Price"]).copy()
    df = df[(df["Size"] != "") & (df["Brand"] != "")]

    pivot = df.pivot_table(index="Size", columns="Brand", values="Price", aggfunc="min").round(2)
    return pivot.sort_index()

def save_all_atomic(out_path: str, rows: list[dict]):
    df = pd.DataFrame(rows)
    pivot = build_pivot_min_price(df)

    tmp = out_path + ".__tmp__.xlsx"
    with pd.ExcelWriter(tmp, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name="Data")
        pivot.to_excel(w, sheet_name="Pivot")

    os.replace(tmp, out_path)  # атомарная замена

    st = os.stat(out_path)
    uniq = df["url"].nunique() if ("url" in df.columns and len(df) > 0) else 0
    print(f"💾 SAVE -> {os.path.abspath(out_path)} | rows={len(df)} | uniq_url={uniq} | mtime={time.ctime(st.st_mtime)}")

def load_existing(out_path: str) -> tuple[list[dict], set[str]]:
    if not os.path.exists(out_path):
        return [], set()
    try:
        df = pd.read_excel(out_path, sheet_name="Data")
        if df is None or df.empty or "url" not in df.columns:
            return [], set()
        df = df.dropna(subset=["url"]).copy()
        df["url"] = df["url"].astype(str).str.strip()
        rows = df.to_dict(orient="records")
        seen = set(df["url"].tolist())
        print(f"📌 Existing file loaded: {out_path} | rows={len(rows)} | seen_urls={len(seen)}")
        return rows, seen
    except Exception as e:
        print(f"⚠️ Can't read existing {out_path}: {e}")
        return [], set()

def load_checkpoint(path: str) -> dict | None:
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def save_checkpoint(path: str, next_index: int, rows_count: int):
    data = {"next_index": int(next_index), "rows_count": int(rows_count), "ts": time.time()}
    tmp = path + ".__tmp__.json"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)
    print(f"✅ Checkpoint -> {os.path.abspath(path)} | next_index={next_index} | rows={rows_count}")

def load_url_cache(path: str) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        arr = json.load(f)
    # только /en/tyres/..., без категорий
    urls = []
    for u in arr:
        if isinstance(u, str) and "/en/tyres/" in u:
            # отсекаем не-товарные разделы (категории)
            # товар обычно содержит размерность "-205-55-r16-" и т.п. или в конце "...-r16-.."
            if re.search(r"-\d{3}-\d{2,3}-r\d{2}\b", u, flags=re.I):
                urls.append(u)
    return urls


# =======================
# RUN
# =======================
print("=== PitstopArabia FULL CATALOG PARSER (Variant A: only актуальные товары с ценой) ===")
print("CWD:", os.getcwd())
print("OUT:", os.path.abspath(OUT_XLSX))
print("CHK:", os.path.abspath(CHECKPOINT_PATH))
print("URLCACHE:", os.path.abspath(URLCACHE_PATH))
print()

if not os.path.exists(URLCACHE_PATH):
    raise FileNotFoundError(f"Нет файла URL-кэша {URLCACHE_PATH}. Сначала нужно собрать urls (sitemap->cache).")

all_product_urls = load_url_cache(URLCACHE_PATH)
print(f"✅ URLs cache loaded: {URLCACHE_PATH} | total_raw={len(json.load(open(URLCACHE_PATH,'r',encoding='utf-8')))} | product_like={len(all_product_urls)}")
print(f"Всего URL товаров к обработке: {len(all_product_urls)}")

all_rows, seen_urls = load_existing(OUT_XLSX)

cp = load_checkpoint(CHECKPOINT_PATH) or {}
start_index = int(cp.get("next_index", 0))

# принудительный “перепроход” проблемного участка
if RESUME_FROM_INDEX is not None:
    start_index = int(RESUME_FROM_INDEX)

print(f"Resume: next_index={start_index} | already_rows={len(all_rows)} | seen_urls={len(seen_urls)}\n")

last_saved = len(all_rows)
processed = 0
added = 0
sk_404 = 0
sk_noprice = 0

for idx in range(start_index, len(all_product_urls)):
    url = all_product_urls[idx]

    # уже есть — пропускаем быстро
    if url in seen_urls:
        processed += 1
        if processed % 500 == 0:
            print(f"--- progress: idx={idx}/{len(all_product_urls)} | processed={processed} | added={added} | uniq={len(seen_urls)} ---")
        continue

    row, status = parse_product_page(url)

    if status == 403:
        print(f"\n🛑 403 detected (likely VPN/off or anti-bot).")
        print(f"Stop here so you can resume later WITHOUT skipping: idx={idx} url={url}\n")
        # сохранение прогресса и выход
        save_all_atomic(OUT_XLSX, all_rows)
        save_checkpoint(CHECKPOINT_PATH, next_index=idx, rows_count=len(all_rows))
        raise SystemExit("Stopped on 403. Turn VPN on / wait, then re-run to continue from checkpoint.")

    if row is None:
        if status == 404:
            sk_404 += 1
        else:
            sk_noprice += 1
        processed += 1
        if processed % 500 == 0:
            print(f"--- progress: idx={idx}/{len(all_product_urls)} | processed={processed} | added={added} | uniq={len(seen_urls)} ---")
        continue

    # добавляем уникальный
    seen_urls.add(url)
    all_rows.append(row)
    processed += 1
    added += 1

    if VERBOSE_ITEMS:
        print(f"OK: {row['brand']} | {row['size']} | {row['model']} | year:{row['year']} | price:{row['price']} | {row['url']}")

    # автосейв каждые N новых строк
    if (len(all_rows) - last_saved) >= SAVE_EVERY_ROWS:
        save_all_atomic(OUT_XLSX, all_rows)
        save_checkpoint(CHECKPOINT_PATH, next_index=idx + 1, rows_count=len(all_rows))
        last_saved = len(all_rows)

    if processed % 500 == 0:
        print(f"--- progress: idx={idx}/{len(all_product_urls)} | processed={processed} | added={added} | uniq={len(seen_urls)} ---")

# финальный сейв
print("\n========================================")
print(f"✅ DONE. rows={len(all_rows)} | uniq_url={len(seen_urls)}")
print(f"skipped: 404={sk_404} | no_price_or_not_product={sk_noprice}")
save_all_atomic(OUT_XLSX, all_rows)
save_checkpoint(CHECKPOINT_PATH, next_index=len(all_product_urls), rows_count=len(all_rows))
print(f"✅ File: {os.path.abspath(OUT_XLSX)} (sheets: Data, Pivot)")
print(f"✅ Checkpoint: {os.path.abspath(CHECKPOINT_PATH)}")


=== PitstopArabia FULL CATALOG PARSER (Variant A: only актуальные товары с ценой) ===
CWD: C:\Users\mochkasov
OUT: C:\Users\mochkasov\pitstoparabia_tyres.xlsx
CHK: C:\Users\mochkasov\pitstop_checkpoint.json
URLCACHE: C:\Users\mochkasov\pitstop_all_product_urls.json

✅ URLs cache loaded: pitstop_all_product_urls.json | total_raw=8669 | product_like=7955
Всего URL товаров к обработке: 7955
📌 Existing file loaded: pitstoparabia_tyres.xlsx | rows=5940 | seen_urls=5940
Resume: next_index=5880 | already_rows=5940 | seen_urls=5940

    GET https://www.pitstoparabia.com/en/tyres/seam-tyre-pearly-n-225-45-r17-94-w -> 200
    GET https://www.pitstoparabia.com/en/tyres/seam-tyre-pearly-n-245-40-r17-95-w -> 200
    GET https://www.pitstoparabia.com/en/tyres/seam-tyre-pearly-n-255-45-r20-105-w -> 200
    GET https://www.pitstoparabia.com/en/tyres/seam-tyre-kasmas-n-225-55-r18-98-v -> 200
    GET https://www.pitstoparabia.com/en/tyres/seam-tyre-jupiter-n-265-45-r20-108-w -> 200
    GET https://www.p